# 00 — API quickstart

A guided tour of the endpoints you *read from* while exploring the
[Explaining Markets](https://explainingmarkets.ai/) competition:

- **`GET /events`** — the events calendar, shown as a clean table and a chart
- **`GET /archive`** — a peek at the historical-data manifest
- **`POST /webhook/test`** — fire a synthetic delivery at your webhook
- **`GET /health`** — your submission's rolling delivery + prediction counters

Everything runs top-to-bottom. With an API key in `.env` you hit the live API;
without one, the calendar section falls back to a small bundled sample so you can
still see the shape of things.

> **Not covered here:** submitting predictions (`POST /predictions`). That places a
> real, scored entry and belongs in a deployed webhook handler — see the starter
> repos, not these examples.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from examples import ApiError, Client, load_config
from examples.frames import events_frame, health_frame, manifest_frame
from examples.plotting import plot_event_calendar
from examples.schemas import CalendarEvent

# Find the repo root whether we're launched from notebooks/ or the repo root.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "sample").is_dir())
SAMPLE = REPO / "data" / "sample"

config = load_config()
print("Mode:    ", "LIVE" if config.is_live else "SAMPLE (no EM_API_KEY — using bundled data)")
print("API base:", config.api_base_url)

## 1. Events calendar — `GET /events`

The calendar lists scheduled and realized events. Each event has a type — an open
string set; currently only `EARNINGS_RELEASE` is emitted, though new types may be
added over time — a timestamp, and its focal assets. We flatten it into a tidy,
sorted table.

In [ ]:
if config.is_live:
    with Client.from_env() as client:
        events = client.events()  # optionally: start_date=..., end_date=...
else:
    raw = json.loads((SAMPLE / "events_sample.json").read_text())
    events = [CalendarEvent.model_validate(e) for e in raw]

events_df = events_frame(events)
print(f"{len(events_df)} events")
events_df.head(15)

A stacked daily bar makes the shape of the calendar obvious at a glance — how many events land each day, split by type.

In [ ]:
fig, ax = plot_event_calendar(events_df, title="Events by day")
plt.show()

## 2. Historical archive — `GET /archive`

The archive is how you get historical events for offline backtesting. Here we just
glance at the manifest; **notebook 01** downloads, caches, and loads the data.

In [ ]:
if config.is_live:
    with Client.from_env() as client:
        manifest = client.archive_manifest()
    display(manifest_frame(manifest))
else:
    print("Sample mode: /archive needs a live EM_API_KEY. See notebook 01 for the full flow.")

## 3. Webhook self-test — `POST /webhook/test`

Fires a synthetic `event_type=TEST` delivery at the webhook URL registered for
your submission. If it's enqueued you get a `202`; the delivery should reach your
endpoint within a few seconds and show up in the health counters below. A failure
here usually means no webhook URL is set, or credentials aren't initialized in the
portal yet.

In [ ]:
if not config.is_live:
    print("Sample mode: firing a test event needs a live EM_API_KEY. Skipping.")
else:
    try:
        with Client.from_env() as client:
            status = client.send_test_event()
        print(f"✓ Test event enqueued (HTTP {status}). It should reach your webhook")
        print("  within a few seconds — watch the health counters in the next section.")
    except ApiError as exc:
        print(f"✗ Test event failed (HTTP {exc.status_code}).")
        print("  Check that your submission has a webhook URL and initialized credentials.")
        print(f"  Details: {exc}")

## 4. Submission health — `GET /health`

Rolling 24-hour counters for outbound webhook deliveries and inbound predictions.
Right after the self-test above, you should see `webhook_n_2xx` (and
`webhook_last_delivery_at`) tick up — proof the round trip works end to end.

In [ ]:
if config.is_live:
    with Client.from_env() as client:
        health = client.health()
    display(health_frame(health))
else:
    print("Sample mode: /health needs a live EM_API_KEY. Skipping.")

## Next steps

- **`01_historical_archive.ipynb`** — download, cache, and load the historical
  archive for offline experiments.
- Ready to compete? Deploy a webhook handler with one of the starter repos and
  submit predictions from there.